# UI 유형 × 숙련도 이원분산분석

효과 코딩 기반 Type III ANOVA를 종속변수별로 한 번 적합합니다. 가정 검정과 변환 후 최종 채택 결과는 04-2번에서 확인합니다.

** p<.05, * .05<p<.10. 경계값 .05와 .10은 별표 없이 표시합니다.

In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

KOREAN_FONT = common.configure_plots()
FIGURE_DIR = common.FIGURE_DIR


## 1. 원자료·척도·집단 검증

In [ ]:
survey = common.load_survey()
DV_ITEMS = common.DV_ITEMS
scores = common.make_scores(survey)
design = common.make_design(survey)
EFFECT_DF = 1
ERROR_DF = len(survey) - 4
observed = pd.crosstab(design["UI"], design["숙련도"]).reindex(index=["A2UI", "Baseline"], columns=["초보자", "숙련자"])
assert observed.notna().all().all() and observed.gt(1).all().all()
assert observed.to_numpy().sum() == len(survey)
display(observed.rename_axis("UI 유형"))


## 2. UI와 숙련도의 독립성 검정

In [ ]:
pearson = stats.chi2_contingency(observed, correction=False)
yates = stats.chi2_contingency(observed, correction=True)
fisher = stats.fisher_exact(observed, alternative="two-sided")

expected = pd.DataFrame(
    pearson.expected_freq,
    index=observed.index,
    columns=observed.columns,
)

independence_tests = pd.DataFrame({
    "검정": ["Pearson χ²", "Yates 보정 χ²", "Fisher 정확검정"],
    "통계량": [pearson.statistic, yates.statistic, fisher.statistic],
    "통계량 종류": ["χ²", "χ²", "오즈비"],
    "자유도": [pearson.dof, yates.dof, np.nan],
    "p": [pearson.pvalue, yates.pvalue, fisher.pvalue],
}).set_index("검정")

print("Pearson 검정의 기대빈도")
display(expected.round(2).rename_axis("UI 유형"))
print("독립성 검정 결과")
display(independence_tests.round(3))


## 3. 원자료 Type III ANOVA

UI·숙련도·상호작용을 함께 포함하며, 각 효과의 자유도는 (1, 36)입니다.

In [ ]:
models = {}
rows = []
for variable in DV_ITEMS:
    model, data, table = common.fit_anova(design, scores[variable])
    models[variable] = model
    rows.append(common.tidy_anova(variable, "원자료", model, table))
raw_results = pd.concat(rows, ignore_index=True)
anova_by_variable = {
    variable: {row["효과"].replace("UI × 숙련도", "UI×숙련도"): (row["F"], row["p"])
               for _, row in raw_results.loc[raw_results["종속변수"].eq(variable)].iterrows()}
    for variable in DV_ITEMS
}
p_with_stars = format_p


In [ ]:
# 1) 최종표의 다중 열 머리글을 정의합니다.
column_pairs = [("자유도", "NumDF"), ("자유도", "DenDF")]
for variable in DV_ITEMS:
    column_pairs.extend([(variable, "F"), (variable, "p")])

# 2) 표시할 행 이름과 계산 결과의 효과 이름을 연결합니다.
effect_rows = [
    ("UItype", "UI"),
    ("숙련도", "숙련도"),
    ("UItype × 숙련도", "UI×숙련도"),
]

# 3) 각 행에 자유도와 종속변수별 F/p를 차례로 넣습니다.
table_values = []
for row_label, effect_key in effect_rows:
    row = [EFFECT_DF, ERROR_DF]

    for variable in DV_ITEMS:
        f_value, p_value = anova_by_variable[variable][effect_key]
        row.extend([f"{f_value:.3f}", p_with_stars(p_value)])

    table_values.append(row)

# 4) 표를 DataFrame으로 만들고 출력합니다.
anova_factor_table = pd.DataFrame(
    table_values,
    index=[row_label for row_label, _ in effect_rows],
    columns=pd.MultiIndex.from_tuples(column_pairs),
)
anova_factor_table.index.name = "Factor"
display(anova_factor_table)


## 4. UI × 숙련도 평균과 95% 신뢰구간

In [ ]:
interaction_rows = []

for variable in DV_ITEMS:
    for ui in ["A2UI", "Baseline"]:
        for proficiency in ["초보자", "숙련자"]:
            mask = design["UI"].eq(ui) & design["숙련도"].eq(proficiency)
            cell_values = scores.loc[mask, variable]
            critical_t = stats.t.ppf(.975, df=len(cell_values) - 1)
            ci_margin = critical_t * stats.sem(cell_values)

            interaction_rows.append({
                "종속변수": variable,
                "UI": ui,
                "숙련도": proficiency,
                "n": len(cell_values),
                "M": cell_values.mean(),
                "95% CI 오차범위": ci_margin,
            })

interaction_plot_data = pd.DataFrame(interaction_rows)
display(interaction_plot_data.round(3))


In [ ]:
proficiency_order = ["초보자", "숙련자"]
x_positions = np.arange(len(proficiency_order))
line_styles = {
    "A2UI": {"color": "black", "linestyle": "-", "marker": "o"},
    "Baseline": {"color": "0.45", "linestyle": "--", "marker": "s"},
}

fig, axes = plt.subplots(2, 4, figsize=(12.0, 6.6), sharey=True)
axes = axes.flatten()

for axis, variable in zip(axes, DV_ITEMS):
    variable_data = interaction_plot_data[interaction_plot_data["종속변수"].eq(variable)]

    for ui in ["A2UI", "Baseline"]:
        ui_data = (
            variable_data[variable_data["UI"].eq(ui)]
            .set_index("숙련도")
            .loc[proficiency_order]
        )
        axis.errorbar(
            x_positions,
            ui_data["M"],
            yerr=ui_data["95% CI 오차범위"],
            linewidth=1.5,
            markersize=5,
            capsize=3,
            label=ui,
            **line_styles[ui],
        )

    axis.set_title(variable)
    axis.set_xticks(x_positions, proficiency_order)
    axis.set_ylim(1, 7)
    axis.set_yticks(range(1, 8))
    axis.grid(axis="y", color="0.88", linewidth=.6)
    axis.spines[["top", "right"]].set_visible(False)

axes[-1].axis("off")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(.5, .98))
fig.suptitle("UI 유형 × 숙련도 상호작용", y=1.02, fontsize=13)
fig.supxlabel("숙련도")
fig.supylabel("평균 점수 (95% CI)")
fig.tight_layout(rect=[.02, .03, 1, .93])

fig.savefig(FIGURE_DIR / "figure_04_interaction_plots.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure_04_interaction_plots.pdf", bbox_inches="tight")
plt.show()
